# P2.4 · Imbalanced binary classification (1 % positives)

**ID:** P2.4 · **Target:** 25 min · **Plan week:** 2 (timed repeat in week 4) · **Part:** P (PyTorch first principles)

**The drill.** About 1 % of examples are positive. Train the same MLP three ways (plain BCE, BCE with `pos_weight`,
and plain BCE with a class-balanced `WeightedRandomSampler`), evaluate them with metrics that are honest under
imbalance, and tune the decision threshold on val. Write the metrics by hand and check them against scikit-learn.

**Must include (from the plan):** `pos_weight` or sampler (here: both), precision/recall/F1, ROC-AUC vs PR-AUC,
threshold tuning. Also: the majority-class baseline, to show why accuracy misleads; initial-loss check (and
prior-bias init); overfit-one-batch; whole-set eval under `eval()` + `no_grad()`; test touched once.

**Rules.** No docs, no LLM. Each solution cell keeps only its spec. The self-check names the variables it expects.

## Sources
- T. Saito, M. Rehmsmeier, *The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary
  Classifiers on Imbalanced Datasets*, PLOS ONE 10(3): e0118432, 2015, DOI 10.1371/journal.pone.0118432 —
  https://doi.org/10.1371/journal.pone.0118432
- J. Davis, M. Goadrich, *The Relationship Between Precision-Recall and ROC Curves*, ICML 2006, pp. 233-240,
  DOI 10.1145/1143844.1143874 — https://doi.org/10.1145/1143844.1143874 (a curve dominates in ROC space iff it
  dominates in PR space; linear interpolation between PR points is wrong).
- Z. C. Lipton, C. Elkan, B. Narayanaswamy, *Thresholding Classifiers to Maximize F1 Score*, ECML-PKDD 2014,
  arXiv:1402.1892 — https://arxiv.org/abs/1402.1892 (the F1-optimal threshold of a calibrated classifier is
  half the optimal F1, so it is far below 0.5 when positives are rare).
- scikit-learn `average_precision_score` (step-wise AP = Σ (Rₙ − Rₙ₋₁) Pₙ, no interpolation), `roc_auc_score`,
  `precision_recall_fscore_support`: https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-and-f-measures
- `torch.nn.BCEWithLogitsLoss` (`pos_weight`) and `torch.utils.data.WeightedRandomSampler` docs:
  https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html ·
  https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.WeightedRandomSampler
- A. Karpathy, *A Recipe for Training Neural Networks* (2019), "init well" (set the final bias to the class
  prior on imbalanced data): https://karpathy.github.io/2019/04/25/recipe/
- J. A. Hanley, B. J. McNeil, *The meaning and use of the area under a ROC curve*, Radiology 143(1):29-36, 1982,
  DOI 10.1148/radiology.143.1.7063747 — https://doi.org/10.1148/radiology.143.1.7063747 (AUC = P(score⁺ > score⁻)).

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from prep import Session, check, check_close, seed_everything

s = Session("P2.4", target_min=25)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. sklearn make_classification(n_samples=60000, n_features=20, n_informative=4, n_redundant=4,
#    weights=[0.99], flip_y=0.0, class_sep=0.7, random_state=0). Print the prevalence.
#    STRATIFIED split 60/20/20 (train_test_split(..., stratify=y, random_state=0) twice). Standardize with train
#    stats. Tensors: X_* float32 (N, 20), y_* float32 (N,) (BCE wants float targets). Also keep numpy y_*_np.
# YOUR CODE HERE


In [ ]:
# 2. METRICS BY HAND (numpy in, python floats out; y in {0,1}).
#    prf(y, yhat) -> (precision, recall, f1); define 0/0 := 0 (sklearn's zero_division=0).
#    roc_auc(y, score): probability that a random positive scores higher than a random negative, ties count 1/2
#      (compare all positive-negative pairs with broadcasting; it's exact, including ties).
#    average_precision(y, score): sort by score descending; AP = mean over positives of precision@(rank of that
#      positive). (Equals sklearn's step-wise AP when scores have no ties.)
# YOUR CODE HERE


In [ ]:
# 3. WHY ACCURACY MISLEADS. Majority-class baseline on val: predict 0 for everyone, score = 0 constant.
#    `baseline = {"acc", "precision", "recall", "f1", "roc_auc", "ap"}` using your functions. Print it.
#    Note what AP equals for a constant (uninformative) score.
# YOUR CODE HERE


In [ ]:
# 4. MODEL + DATA PIPELINES.
#    `MLP`: 20 -> 64 -> 64 -> 1, ReLU, returns (B,) logits. make_model(prior_bias=False): if True, set the last
#    layer's bias to log(p / (1 - p)) with p = train prevalence ("init well").
#    plain_dl: train, batch 256, shuffle=True.
#    sampler_dl: same data, batch 256, WeightedRandomSampler with per-EXAMPLE weight 1 / count(class of example),
#      num_samples = len(train), replacement=True (each batch then has ~50 % positives).
#    `pos_weight` = tensor(n_neg / n_pos) for BCEWithLogitsLoss.
# YOUR CODE HERE


In [ ]:
# 5. SANITY CHECKS. On the val set, BCE of a fresh model:
#    `init_loss_default` (default init, expect ≈ ln 2) and `init_loss_prior` (prior-bias init, expect ≈ the binary
#    entropy H(p) = -p ln p - (1-p) ln(1-p)). Then overfit ONE sampler batch (256, ~half positives) with a fresh
#    model, Adam lr=1e-2, 300 steps -> `overfit_loss` (< 0.05).
# YOUR CODE HERE


In [ ]:
# 6. TRAIN THREE WAYS. `train(mode, epochs=15) -> model`, with torch.manual_seed(0) at the start, AdamW lr=1e-3:
#      "plain":      plain_dl,   BCEWithLogitsLoss(),                      prior-bias init
#      "pos_weight": plain_dl,   BCEWithLogitsLoss(pos_weight=pos_weight), default init
#      "sampler":    sampler_dl, BCEWithLogitsLoss(),                      default init
#    `scores(model, X) -> numpy probabilities` (eval, no_grad, sigmoid). models = {mode: model}.
#    val_probs = {mode: probabilities on val}.
# YOUR CODE HERE


In [ ]:
# 7. COMPARE AT THRESHOLD 0.5 (val). For each model: accuracy, precision, recall, F1 at p >= 0.5, ROC-AUC, AP.
#    Store `val_at_05 = {mode: {"acc", "precision", "recall", "f1", "roc_auc", "ap"}}`; print a table
#    (include the majority baseline row). What do the ranking metrics (AUC, AP) say vs the thresholded ones?
# YOUR CODE HERE


In [ ]:
# 8. THRESHOLD TUNING ON VAL. best_threshold(y, p) -> (t, f1): the threshold maximizing F1 of (p >= t), searching
#    over all distinct scores. Vectorize: sort p descending, cumulative TP/FP at each cut, F1 = 2TP / (2TP + FP + FN);
#    with tied scores only the LAST index of each tie group is a valid cut.
#    `thresholds = {mode: t}` from val; then (test touched once) `test_tuned = {mode: report(y_test, p_test, t)}`
#    and `test_at_05` likewise at 0.5. Print both tables.
# YOUR CODE HERE


In [ ]:
# 9. CURVES (test). Left: ROC curves of the three models + the chance diagonal. Right: PR curves + a horizontal
#    line at the prevalence (the PR chance level). You may use sklearn.metrics.roc_curve / precision_recall_curve
#    for plotting only.
# YOUR CODE HERE


## Self-check
Expects: `y_train`, `prf`, `roc_auc`, `average_precision`, `baseline`, `pos_weight`, `sampler_dl`,
`init_loss_default`, `init_loss_prior`, `overfit_loss`, `models`, `val_probs`, `val_at_05`, `best_threshold`,
`thresholds`, `test_tuned`.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, precision_recall_fscore_support, roc_auc_score

rng = np.random.default_rng(0)
yy = (rng.random(2000) < 0.05).astype(int)
ss = rng.random(2000) + 0.6 * yy                 # continuous scores, informative
hard = (ss > 0.9).astype(int)
check("targets are float for BCE", y_train.dtype == torch.float32)
check_close("prf == sklearn", prf(yy, hard), precision_recall_fscore_support(yy, hard, average="binary", zero_division=0)[:3])
check_close("prf zero division -> 0", prf(yy, np.zeros_like(yy)), [0.0, 0.0, 0.0])
check_close("roc_auc == sklearn", roc_auc(yy, ss), roc_auc_score(yy, ss))
check_close("roc_auc with ties == sklearn", roc_auc(yy, np.round(ss, 1)), roc_auc_score(yy, np.round(ss, 1)))
check_close("average_precision == sklearn", average_precision(yy, ss), average_precision_score(yy, ss))
check("majority baseline: high accuracy, zero F1", baseline["acc"] > 0.98 and baseline["f1"] == 0.0, str(baseline))
check_close("constant score: AUC = 0.5", baseline["roc_auc"], 0.5)
check_close("pos_weight = n_neg / n_pos", float(pos_weight), float((y_train == 0).sum() / (y_train == 1).sum()), rtol=1e-3)
frac = torch.cat([yb for _, yb in sampler_dl]).mean().item()
check("sampler balances classes (~50 % positives per epoch)", 0.4 < frac < 0.6, f"got {frac:.3f}")
check("initial loss default init ~ ln 2", abs(init_loss_default - math.log(2)) < 0.15, f"got {init_loss_default:.3f}")
p0 = y_train.mean().item()
check("initial loss with prior bias ~ H(p)", abs(init_loss_prior - (-p0 * math.log(p0) - (1 - p0) * math.log(1 - p0))) < 0.02, f"got {init_loss_prior:.4f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
for m in models:
    t, f = best_threshold(y_val_np, val_probs[m])
    cand = np.unique(np.r_[val_probs[m], 0.5])
    brute = max(f1_score(y_val_np, val_probs[m] >= c) for c in cand[::max(1, len(cand) // 400)])
    check(f"{m}: tuned val F1 == sklearn F1 at t", abs(f - f1_score(y_val_np, val_probs[m] >= t)) < 1e-9)
    check(f"{m}: tuned F1 >= F1 at 0.5 and >= sub-sampled brute force", f >= val_at_05[m]["f1"] - 1e-9 and f >= brute - 1e-9)
    check(f"{m}: ROC-AUC > 0.90 on test", test_tuned[m]["roc_auc"] > 0.90, f"got {test_tuned[m]['roc_auc']:.3f}")
    check(f"{m}: AP well above prevalence on test", test_tuned[m]["ap"] > 10 * y_test_np.mean(), f"got {test_tuned[m]['ap']:.3f}")
check("pos_weight model needs a higher threshold than the plain model",
      thresholds["pos_weight"] > thresholds["plain"], f"{thresholds}")

## Say it out loud (P3)
1. The majority baseline has ~99 % accuracy. Why is accuracy the wrong headline metric here, and what would you report?
2. ROC-AUC barely separates the three models; AP/PR curves separate them more. Why (what does FPR hide when negatives are 99 %)?
3. What does `pos_weight = n_neg/n_pos` do to the loss and gradients? What does it do to the *calibration* of the probabilities?
4. Sampler vs `pos_weight`: same expected gradient? What are the practical differences (variance, epochs, overfitting)?
5. Why is the F1-optimal threshold of the plain model far below 0.5, and why must it be tuned on val, not test?
6. What is the chance level of AP, and why isn't it 0.5? Why shouldn't you linearly interpolate a PR curve?
7. Why set the final bias to log(p/(1−p)) at init? What does the initial loss become?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up